# Simplified RLHF Loop

Companion notebook for the [Reinforcement Learning from Human Feedback](https://codemia.io/courses/llm_inside_and_out/rlhf) lesson on Codemia.

**What you will build**: a hand-rolled RLHF loop with three components: a tiny reward model trained on preference pairs, a frozen reference policy, and a PPO-style update on a small language model. You will watch the KL penalty in action and see what happens when you set beta too low.

**Time**: ~60-90 min

**GPU required**: yes (T4 free tier works)

Note: this is a teaching implementation, not a production-quality RLHF stack. Production code uses libraries like `trl` for the PPO loop and a heavily-engineered reward model. The point here is to see the moving parts in 200 lines of Python so the formulas in the lesson stop being abstract.

## Setup

In [ ]:
%pip install -q torch transformers datasets matplotlib

In [ ]:
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using {device}')

## Concept recap

RLHF turns a frozen base language model into a preference-aligned model in three stages:

1. **Reward model** ($r_\phi$): trained on pairs `(prompt, chosen, rejected)` so it scores `chosen > rejected`. This is the only component that touches human preferences directly.
2. **Reference policy** ($\pi_\text{ref}$): an immutable copy of the base model. The KL penalty against this reference is what stops the policy from collapsing into reward-hacking gibberish.
3. **Policy** ($\pi_\theta$): the model being trained. At each step, sample a response, score it with the reward model, and update via PPO with a KL penalty: `total_reward = r(x, y) - β · KL(π_θ || π_ref)`.

The single most important hyperparameter is `β`. Too low: the policy goes off the rails because it has unlimited freedom to chase reward. Too high: the policy can't move and stays identical to the reference. We will deliberately tune β too low at the end to see the failure mode.

## Step 1: load the base model and a hand-crafted preference dataset

We use `distilgpt2` (82M parameters) so the whole loop runs on a free Colab GPU. The preference dataset is hand-written: for each prompt we have one `chosen` (formal, helpful) and one `rejected` (terse, dismissive) response. This is the same shape as HH-RLHF or UltraFeedback, just smaller.

In [ ]:
model_name = 'distilgpt2'
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

policy = GPT2LMHeadModel.from_pretrained(model_name).to(device)
ref_model = copy.deepcopy(policy).eval()
for p in ref_model.parameters():
    p.requires_grad = False

print(f'Policy params: {sum(p.numel() for p in policy.parameters()):,}')
print(f'Reference params (frozen): {sum(p.numel() for p in ref_model.parameters()):,}')

In [ ]:
# Hand-crafted preference dataset. In real RLHF this would be 50K-1M pairs.
preferences = [
    {
        'prompt': 'Q: How do I learn Python?\nA:',
        'chosen': ' Start with a free interactive course like Codecademy, then build small projects to practice.',
        'rejected': ' Just google it.',
    },
    {
        'prompt': 'Q: What is recursion?\nA:',
        'chosen': ' Recursion is when a function calls itself with a smaller version of the problem until a base case stops it.',
        'rejected': ' Look it up.',
    },
    {
        'prompt': 'Q: Should I learn Rust?\nA:',
        'chosen': ' If you want a fast, safe systems language with great tooling, yes. It has a steep learning curve but the payoff is real.',
        'rejected': ' Maybe, who cares.',
    },
    {
        'prompt': 'Q: How do neural networks learn?\nA:',
        'chosen': ' They adjust their weights through gradient descent to minimize a loss function on labeled training data.',
        'rejected': ' Magic.',
    },
    {
        'prompt': 'Q: What is a database index?\nA:',
        'chosen': ' A data structure (often a B-tree) that lets the database find rows matching a query without scanning the entire table.',
        'rejected': ' Idk.',
    },
]
print(f'{len(preferences)} preference pairs')

## Step 2: train the reward model

The reward model is the policy's last hidden state passed through a linear head that produces a scalar. We train it with the Bradley-Terry loss: `loss = -log σ(r(chosen) - r(rejected))`. After training, `r(chosen) > r(rejected)` for the pairs in the dataset.

In [ ]:
class RewardModel(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        # Reuse the base model body and add a fresh scalar head
        self.body = copy.deepcopy(base_model.transformer)
        self.head = nn.Linear(self.body.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        # Take the hidden state at the LAST non-padding position
        out = self.body(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = out.last_hidden_state
        # Find the last real token per row
        last_idx = attention_mask.sum(dim=1) - 1
        pooled = last_hidden[torch.arange(last_hidden.size(0)), last_idx]
        return self.head(pooled).squeeze(-1)

reward_model = RewardModel(policy).to(device)
rm_opt = Adam(reward_model.parameters(), lr=1e-5)

In [ ]:
def encode(prompt, response):
    text = prompt + response
    enc = tokenizer(text, return_tensors='pt', padding='max_length',
                    truncation=True, max_length=64)
    return {k: v.to(device) for k, v in enc.items()}

# Train the reward model for a few epochs on the preference pairs
rm_losses = []
for epoch in range(80):
    epoch_loss = 0.0
    for ex in preferences:
        chosen_enc = encode(ex['prompt'], ex['chosen'])
        rejected_enc = encode(ex['prompt'], ex['rejected'])
        r_chosen = reward_model(**chosen_enc)
        r_rejected = reward_model(**rejected_enc)
        # Bradley-Terry: encourage r(chosen) > r(rejected)
        loss = -F.logsigmoid(r_chosen - r_rejected).mean()
        rm_opt.zero_grad()
        loss.backward()
        rm_opt.step()
        epoch_loss += loss.item()
    rm_losses.append(epoch_loss / len(preferences))
    if epoch % 10 == 0:
        print(f'epoch {epoch:3d}  rm_loss {rm_losses[-1]:.4f}')

plt.plot(rm_losses)
plt.xlabel('epoch'); plt.ylabel('Bradley-Terry loss'); plt.title('Reward model training')
plt.show()

In [ ]:
# Sanity check: reward model should score chosen > rejected on every pair
reward_model.eval()
for ex in preferences:
    rc = reward_model(**encode(ex['prompt'], ex['chosen'])).item()
    rr = reward_model(**encode(ex['prompt'], ex['rejected'])).item()
    print(f"chosen {rc:+.2f}  rejected {rr:+.2f}  {'OK' if rc > rr else 'BAD'}  ({ex['prompt'][:30]}...)")

## Step 3: the PPO-style RL loop with KL penalty

For each prompt, we sample a response from the policy, score it with the reward model, compute the KL divergence against the reference policy, and update with the combined reward. We use a simple REINFORCE-style update (real PPO adds clipping for stability, which we skip for clarity).

In [ ]:
def sample_response(model, prompt, max_new_tokens=20, temperature=1.0):
    """Generate one response and return ids + per-token log-probs."""
    enc = tokenizer(prompt, return_tensors='pt').to(device)
    input_ids = enc['input_ids']
    log_probs = []
    for _ in range(max_new_tokens):
        with torch.no_grad() if not model.training else torch.enable_grad():
            logits = model(input_ids).logits[:, -1, :] / temperature
        dist = torch.distributions.Categorical(logits=logits)
        next_tok = dist.sample()
        log_probs.append(dist.log_prob(next_tok))
        input_ids = torch.cat([input_ids, next_tok.unsqueeze(0)], dim=1)
        if next_tok.item() == tokenizer.eos_token_id:
            break
    return input_ids, torch.stack(log_probs)

In [ ]:
def compute_kl(policy_logp, ref_logp):
    # Per-token KL approximation, summed over the response
    return (policy_logp - ref_logp).sum()

def ppo_step(prompt, beta=0.1):
    policy.train()
    ids, policy_logp = sample_response(policy, prompt)

    # Score the full response with the reward model
    attn = torch.ones_like(ids)
    reward = reward_model(input_ids=ids, attention_mask=attn).item()

    # Reference log-probs for the same generated tokens
    with torch.no_grad():
        ref_logits = ref_model(ids).logits[:, :-1, :]
        ref_targets = ids[:, 1:]
        ref_logp = F.log_softmax(ref_logits, dim=-1).gather(
            -1, ref_targets.unsqueeze(-1)).squeeze(-1).sum()

    kl = compute_kl(policy_logp.sum(), ref_logp)
    # Total signal: reward minus KL penalty
    advantage = reward - beta * kl.item()
    # REINFORCE update: maximize log_prob * advantage
    loss = -(policy_logp.sum() * advantage)
    return loss, reward, kl.item()

## Step 4: train with a sane β

First we run the loop with `β = 0.1`, which is in the typical RLHF range. The reward should rise and the KL should grow modestly.

In [ ]:
policy_opt = Adam(policy.parameters(), lr=1e-5)
rewards_sane, kls_sane = [], []

for step in range(60):
    prompt = preferences[step % len(preferences)]['prompt']
    loss, reward, kl = ppo_step(prompt, beta=0.1)
    policy_opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
    policy_opt.step()
    rewards_sane.append(reward)
    kls_sane.append(kl)
    if step % 10 == 0:
        print(f'step {step:3d}  reward {reward:+.2f}  kl {kl:+.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(rewards_sane); axes[0].set_title('Reward (β=0.1)')
axes[1].plot(kls_sane); axes[1].set_title('KL vs reference (β=0.1)')
for ax in axes: ax.set_xlabel('step')
plt.tight_layout(); plt.show()

## Step 5: what happens when β is too low

Set `β = 0.001` (100x smaller). The KL penalty barely constrains the policy, so it is free to drift far from the reference and chase reward. Watch the KL explode while the reward grows. This is the failure mode RLHF practitioners are constantly defending against.

In [ ]:
# Reset policy to a fresh copy of the reference
policy = copy.deepcopy(ref_model)
for p in policy.parameters():
    p.requires_grad = True
policy_opt = Adam(policy.parameters(), lr=1e-5)

rewards_low, kls_low = [], []
for step in range(60):
    prompt = preferences[step % len(preferences)]['prompt']
    loss, reward, kl = ppo_step(prompt, beta=0.001)
    policy_opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
    policy_opt.step()
    rewards_low.append(reward)
    kls_low.append(kl)
    if step % 10 == 0:
        print(f'step {step:3d}  reward {reward:+.2f}  kl {kl:+.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(rewards_sane, label='β=0.1 (sane)')
axes[0].plot(rewards_low, label='β=0.001 (too low)')
axes[0].set_title('Reward'); axes[0].legend()
axes[1].plot(kls_sane, label='β=0.1')
axes[1].plot(kls_low, label='β=0.001')
axes[1].set_title('KL vs reference'); axes[1].legend()
for ax in axes: ax.set_xlabel('step')
plt.tight_layout(); plt.show()

## Variations

The interesting hyperparameters to sweep:

- `β`: try 0.01, 0.1, 0.5, 1.0. Find the sweet spot for your reward model.
- `temperature` during sampling: higher means more exploration.
- Reward model architecture: a separate small model vs the head you trained here.
- Number of samples per prompt: in real PPO you sample multiple responses and use the relative ranking.

## What to try next

- Replace this hand-rolled loop with TRL's `PPOTrainer` and observe the differences (clipping, value head, advantage normalization, KL coefficient adaptation).
- Use a real preference dataset like Anthropic's HH-RLHF subset on Hugging Face.
- Train a separate reward model rather than one that shares weights with the base policy.
- Implement the DPO objective on the same preference data and compare against RLHF (see the DPO companion notebook in this repo).